In [1]:
import pandas as pd
from ir_datasets_longeval import load, irds_registry

In [29]:
def filter_docs_not_in_qrels(qrels_path, irds):
    def get_doc(doc_id):
        try:
            return store.get(doc_id)
        except KeyError:
            return None
        
    dataset = load(irds)
    store = dataset.docs_store()
    
    qrels = pd.read_csv(qrels_path, sep=" ", names=["query_id", "iter", "doc_id", "relevance"])
    
    qrels["doc"] = qrels.apply(lambda x: get_doc(str(x["doc_id"])), axis=1)
    
    print("# qrels original", qrels.shape[0])
    qrels = qrels.dropna()
    print("# qrels after filtering", qrels.shape[0])
    return qrels[["query_id", "iter", "doc_id", "relevance"]]


def check_valid_queries(qrels):
    # valid: queries with at least one relevant doc
    queries_valid = qrels.dropna().groupby("query_id")["relevance"].sum()
    print("# queries original", queries_valid.shape[0])
    
    valid_queries = queries_valid[queries_valid >= 1]
    print("# queries valid", valid_queries.shape[0])
    return valid_queries.index

# Train

In [55]:
irds = f"longeval-sci-2026/03-05-train"
qrels_path = f"data/qrels-snapshot-train-dctr.txt"
qrels = filter_docs_not_in_qrels(qrels_path, irds)
valid_qids_train = check_valid_queries(qrels)

# qrels original 10406
# qrels after filtering 8772
# queries original 100
# queries valid 100


In [56]:
queries = pd.read_csv("data/queries-train.tsv", sep="\t", names=["query_id", "query"])
queries = queries[queries["query_id"].isin(valid_qids_train)]
queries.to_csv(f"data-cleaned/queries-snapshot-train.tsv", sep="\t", index=False, header=False)

## Test

In [57]:
valid_qids_test = set()
for snapshot, irds_snapshot in [("1", "03-05"), ("2", "06-08"), ("3", "09-11")]:
    irds = f"longeval-sci-2026/{irds_snapshot}"
    qrels_path = f"data/qrels-snapshot-{snapshot}-dctr.txt"
    qrels = filter_docs_not_in_qrels(qrels_path, irds)
    print()
    valid_qids = set(check_valid_queries(qrels))
    valid_qids_test.update(valid_qids)

# qrels original 24080
# qrels after filtering 21247

# queries original 149
# queries valid 147
# qrels original 18832
# qrels after filtering 16116

# queries original 100
# queries valid 96
# qrels original 11338
# qrels after filtering 9878

# queries original 135
# queries valid 133


In [60]:
queries = pd.read_csv("data/queries-test.tsv", sep="\t", names=["query_id", "query"])
queries = queries[queries["query_id"].isin(valid_qids_test)]
queries.to_csv(f"data-cleaned/queries-snapshot-test.tsv", sep="\t", index=False, header=False)

## qrels

In [ ]:
def clean(snapshot):
    if snapshot == "train":
        query_name = "train"
    else:
        query_name = "test"
    queries = pd.read_csv(f"data-cleaned/queries-snapshot-{query_name}.tsv", sep="\t", names=["query_id", "query"])
    
    qrels_raw = pd.read_csv(f"data/qrels-snapshot-{snapshot}-raw.txt", sep=" ", names=["query_id", "iter", "doc_id", "relevance"])
    qrels_raw = qrels_raw[qrels_raw["query_id"].isin(queries["query_id"])]
    qrels_raw.to_csv(f"data-cleaned/qrels-snapshot-{snapshot}-raw.txt", sep=" ", index=False, header=False)
    
    qrels_dctr = pd.read_csv(f"data/qrels-snapshot-{snapshot}-dctr.txt", sep=" ", names=["query_id", "iter", "doc_id", "relevance"])
    qrels_dctr = qrels_dctr[qrels_dctr["query_id"].isin(queries["query_id"])]
    qrels_dctr.to_csv(f"data-cleaned/qrels-snapshot-{snapshot}-dctr.txt", sep=" ", index=False, header=False)

In [65]:
for snapshot in ["train", "1", "2", "3"]:
    clean(snapshot)

## Check

In [66]:
base_dir = "data-cleaned"
queries_train = pd.read_csv(f"{base_dir}/queries-snapshot-train.tsv", sep="\t", names=["query_id", "query"])
queries_train_ids = set(queries_train["query_id"].unique())
queries_test = pd.read_csv(f"{base_dir}/queries-snapshot-test.tsv", sep="\t", names=["query_id", "query"])
queries_test_ids = set(queries_test["query_id"].unique())

for snapshot in ["train", "1", "2", "3"]:    
    qrels_raw = pd.read_csv(f"{base_dir}/qrels-snapshot-{snapshot}-raw.txt", sep=" ", names=["query_id", "iter", "doc_id", "relevance"])
    qrels_ids = set(qrels_raw["query_id"].unique())
    assert qrels_ids.issubset(queries_train_ids) or qrels_ids.issubset(queries_test_ids), f"Qrels {snapshot} has query ids not in queries"
    qrels_dctr = pd.read_csv(f"{base_dir}/qrels-snapshot-{snapshot}-dctr.txt", sep=" ", names=["query_id", "iter", "doc_id", "relevance"])
    assert set(qrels_dctr["query_id"].unique()).issubset(queries_train_ids) or set(qrels_dctr["query_id"].unique()).issubset(queries_test_ids), f"Qrels {snapshot} has query ids not in queries"
